<a href="https://colab.research.google.com/github/women-in-ai-ireland/June-2024-Group-002/blob/embeddings/Embeddings_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embeddings

Embeddings using [Instructor Embedding](https://huggingface.co/hkunlp)

#### Prepare environment

In [2]:
# install libraries
!pip install langchain -U langchain-community faiss-cpu langchain-openai tiktoken unstructured selenium newspaper3k textstat

!pip install sentence-transformers==2.2.2
!pip install InstructorEmbedding

!pip install pypdf

!pip install torch==2.1.0
!pip install accelerate==0.21.0 transformers==4.31.0 tokenizers==0.13.3
!pip install bitsandbytes==0.40.0 einops==0.6.1
!pip install xformers==0.0.22.post7

!pip install accelerate bitsandbytes

In [ ]:
# import libraries

import pickle
from google.colab import userdata #to load google colab secret.
import os
from google.colab import drive
import getpass
from pypdf import PdfReader

from langchain.document_loaders import WebBaseLoader, UnstructuredURLLoader, NewsURLLoader, SeleniumURLLoader
from langchain_openai import OpenAI
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import PyPDFLoader

# InstructorEmbedding
from InstructorEmbedding import INSTRUCTOR
from langchain.embeddings import HuggingFaceInstructEmbeddings

from transformers import AutoTokenizer, AutoModelForCausalLM

from langchain.chains import RetrievalQA


In [ ]:
# access HF secret
hf_token = userdata.get('HF_TOKEN')

In [ ]:
# set open ai api key
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
# connect to Google Drive
drive.mount('/content/gdrive', force_remount=True)
root_dir = "/content/gdrive/MyDrive/WAI_project/"

In [ ]:
# set local path to store embeddings ***replace with SingleStore, AWS or similar
embedding_store_path = f"{root_dir}/embedding_store"

In [ ]:
# defines the parameters to use in the recursive text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 100,
    length_function = len,
)

#### Helper functions

In [ ]:
# Step 1: Ingestion and Chunking

def ingest_and_chunk_pdfs():
    """
    Ingests PDF documents and chunks the content into smaller segments.

    Returns:
    - list: List of text chunks.
    """

    loader = DirectoryLoader(f"{root_dir}/data/", glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    texts = text_splitter.split_documents(documents)
    return texts


In [ ]:
# Function to store embeddings
def store_embeddings(docs, embeddings, store_name, path):
    """
    Stores embeddings in FAISS format and saves to a pickle file.

    Args:
    - docs (list): List of documents.
    - embeddings: Embedding model.
    - store_name (str): Name of the embedding store.
    - path (str): Path to the directory where embeddings will be stored.
    """
    vector_store = FAISS.from_documents(docs, embeddings)
    with open(os.path.join(path, f"faiss_{store_name}.pkl"), "wb") as f:
        pickle.dump(vector_store, f)

In [ ]:
# Function to load embeddings
def load_embeddings(store_name, path):
    """
    Loads embeddings from a pickle file.

    Args:
    - store_name (str): Name of the embedding store.
    - path (str): Path to the directory where embeddings are stored.

    Returns:
    - vector_store: Loaded FAISS vector store.
    """
    with open(os.path.join(path, f"faiss_{store_name}.pkl"), "rb") as f:
        vector_store = pickle.load(f)
    return vector_store

In [ ]:
# Function to initialize Hugging Face Instruct Embeddings
def initialize_huggingface_embeddings(model_name="hkunlp/instructor-xl", device="cuda"):
    """
    Initializes Hugging Face Instruct Embeddings model.

    Args:
    - model_name (str): Name of the Hugging Face model.
    - device (str): Device to run the model on.

    Returns:
    - embeddings: Initialized Hugging Face Instruct Embeddings model.
    """
    return HuggingFaceInstructEmbeddings(model_name=model_name, model_kwargs={"device": device})


In [ ]:
# get embeddings from local db
retriever = db_instructEmbedd.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("Who is Fela Kuti?")
docs[0]

In [ ]:
# Load Llama2 and returns the most similar vector embedding based on the given prompt
# TO DO: use local embeddings along llama2
def load_llm(model_name='meta-llama/Llama-2-7b-chat-hf'):

  model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", load_in_4bit=True,  use_auth_token = hf_token)
  tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, use_auth_token=hf_token)


In [4]:
# Load Llama2 and returns the most similar vector embedding based on the given prompt
# TO DO: use local embeddings along llama2
def load_llm(model_name='meta-llama/Llama-2-7b-chat-hf'):
  """
    Load Llama2 along the model weights.

    Args:
    - model_name (str): Name of the Hugging Face model.

    Returns:
    - tokenizer: Initialized Llama2 model with weights
  """
  model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", load_in_4bit=True,  use_auth_token = hf_token)
  tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, use_auth_token=hf_token)

  return tokenizer

##### Main API function:

In [ ]:
def main(): # for the API, replace with   if __name__ == "__main__":

  # initialize embedding model
  initialize_huggingface_embeddings()

  # Ingest and chunk PDFs
  texts = ingest_and_chunk_pdfs()

  # store embeddings in a local folder
  store_embeddings(texts,
                  instructor_embeddings,
                  store_name='instructEmbeddings',
                  path=embedding_store_path)

  # load the embeddings
  db_instructEmbedd = load_embeddings(store_name='instructEmbeddings',
                                      path=embedding_store_path)

  # query llama2 without using the local embeddings
  # TO DO: receive the promt from user input
  tokenizer = load_llm()
  prompt = "Who is Fela Kuti?"
  model_inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
  output = model.generate(**model_inputs)
  print(tokenizer.decode(output[0], skip_special_tokens=True))


ValueError: 
                        Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit
                        the quantized model. If you want to dispatch the model on the CPU or the disk while keeping
                        these modules in 32-bit, you need to set `load_in_8bit_fp32_cpu_offload=True` and pass a custom
                        `device_map` to `from_pretrained`. Check
                        https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu
                        for more details.
                        

#### TEST CODE   
Try Microsoft Phi3, a tiny llm
This code was intended to replace Llama2 for a smaller model, but it is still not working on Google Colab

Results: Not enough memory to run in Google Colab

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

torch.random.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Can you provide ways to eat combinations of bananas and dragonfruits?"},
    {"role": "assistant", "content": "Sure! Here are some ways to eat bananas and dragonfruits together: 1. Banana and dragonfruit smoothie: Blend bananas and dragonfruits together with some milk and honey. 2. Banana and dragonfruit salad: Mix sliced bananas and dragonfruits together with some lemon juice and honey."},
    {"role": "user", "content": "What about solving an 2x + 3 = 7 equation?"},
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "temperature": 0.0,
    "do_sample": False,
}

output = pipe(messages, **generation_args)
print(output[0]['generated_text'])


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py:   0%|          | 0.00/73.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ImportError: Using `low_cpu_mem_usage=True` or a `device_map` requires Accelerate: `pip install accelerate`